In [7]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [8]:
duplicates = train_df[train_df.duplicated(['text'], keep=False)]
conflicts = duplicates.groupby('text')['target'].nunique()
conflicting_texts = conflicts[conflicts > 1].index

train_df_clean = train_df[~train_df['text'].isin(conflicting_texts)].copy()
train_df_clean = train_df_clean.drop_duplicates(subset=['text'])

print(f"Rozmiar przed: {len(train_df)}, Rozmiar po czyszczeniu: {len(train_df_clean)}")

Rozmiar przed: 7613, Rozmiar po czyszczeniu: 7485


In [9]:
import re
def bertweet_preprocess(text):
    # Zamiana linków na token rozpoznawany przez BERTweet
    text = re.sub(r'http\S+', 'HTTPURL', text)
    # Zamiana oznaczeń użytkowników na token @USER
    text = re.sub(r'@\w+', '@USER', text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df_clean['clean_text'] = train_df_clean['text'].apply(bertweet_preprocess)

In [16]:
test_df['clean_text'] = test_df['text'].apply(bertweet_preprocess)

test_dataset = Dataset.from_pandas(test_df[['id', 'clean_text']].rename(columns={'clean_text': 'text'}))

In [11]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [12]:
train_data, val_data = train_test_split(
    train_df_clean[['clean_text', 'target']].rename(columns={'clean_text': 'text', 'target': 'label'}),
    test_size=0.2,
    random_state=42,
    stratify=train_df_clean['target']
)

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

In [13]:
MODEL_NAME = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results_bertweet",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=100,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Ewaluacja
print("\nEwaluacja BERTweet na zbiorze walidacyjnym:")
val_predictions = trainer.predict(tokenized_val)

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Map:   0%|          | 0/5988 [00:00<?, ? examples/s]

... (5 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.421801,0.829659,0.780362
2,0.428682,0.415946,0.834335,0.800000
3,0.296782,0.473751,0.833667,0.795062


model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            


Ewaluacja BERTweet na zbiorze walidacyjnym:


In [14]:
from sklearn.metrics import classification_report

labels = val_predictions.label_ids
preds = np.argmax(val_predictions.predictions, axis=-1)

print("Accuracy:", round(accuracy_score(labels, preds), 4))
print("F1 Score:", round(f1_score(labels, preds), 4))
print("\nRaport klasyfikacji:\n", classification_report(labels, preds))

# Analiza błędów
val_analysis = val_data.copy()
val_analysis['pred'] = preds

# Fałszywe alarmy
false_positives = val_analysis[(val_analysis['label'] == 0) & (val_analysis['pred'] == 1)]
print(f"\nLiczba fałszywych alarmów (FP): {len(false_positives)}")
print("Przykłady FP:")
for text in false_positives['text'].head(3):
    print("-", text)

# Pominięte katastrofy
false_negatives = val_analysis[(val_analysis['label'] == 1) & (val_analysis['pred'] == 0)]
print(f"\nLiczba pominiętych katastrof (FN): {len(false_negatives)}")
print("Przykłady FN:")
for text in false_negatives['text'].head(3):
    print("-", text)

Accuracy: 0.8337
F1 Score: 0.7951

Raport klasyfikacji:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86       859
           1       0.84      0.76      0.80       638

    accuracy                           0.83      1497
   macro avg       0.83      0.82      0.83      1497
weighted avg       0.83      0.83      0.83      1497


Liczba fałszywych alarmów (FP): 94
Przykłady FP:
- Tales of the #trees #deep water loving #Lake Tahoe. And no #forest fires HTTPURL
- @USER lets hope it's a upper class white mass murderer....''' Mmmm
- Is your team ready for a natural disaster a violent client or power outage? Contact Ready Vet to design... HTTPURL

Liczba pominiętych katastrof (FN): 155
Przykłady FN:
- Senators calling for emergency housing: Boxer Feinstein back plan to move #homeless vets to VA campus HTTPURL
- @USER went down I was beaten and blown up. Then next thing I know Ra Al Ghul brought me back to life and I escaped and for a---
-

## ensemble

In [18]:
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

In [19]:
from sklearn.model_selection import StratifiedKFold
from scipy.special import softmax

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
test_probs = np.zeros((len(test_dataset), 2))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df_clean, train_df_clean['target'])):
    print(f"\n--- Trenowanie Foldu {fold + 1}/3 ---")

    train_fold = train_df_clean.iloc[train_idx][['clean_text', 'target']].rename(columns={'clean_text': 'text', 'target': 'label'})
    val_fold = train_df_clean.iloc[val_idx][['clean_text', 'target']].rename(columns={'clean_text': 'text', 'target': 'label'})

    tokenized_train_fold = Dataset.from_pandas(train_fold).map(tokenize_function, batched=True)
    tokenized_val_fold = Dataset.from_pandas(val_fold).map(tokenize_function, batched=True)

    model_fold = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    trainer_fold = Trainer(
        model=model_fold,
        args=training_args,
        train_dataset=tokenized_train_fold,
        eval_dataset=tokenized_val_fold,
        processing_class=tokenizer,
        data_collator=data_collator
    )

    trainer_fold.train()

    preds_output = trainer_fold.predict(tokenized_test)
    test_probs += softmax(preds_output.predictions, axis=1) / skf.n_splits

final_preds = np.argmax(test_probs, axis=1)

# zapis
submission = pd.DataFrame({'id': test_df['id'], 'target': final_preds})
submission.to_csv('submission_kfold.csv', index=False)
print("Plik submission_kfold.csv gotowy do wysłania na Kaggle!")


--- Trenowanie Foldu 1/3 ---


Map:   0%|          | 0/4990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2495 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,0.428013
2,0.435350,0.409690
3,0.435350,0.446818


... (9 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,0.418671
2,0.431756,0.416192
3,0.431756,0.463269


Plik submission_kfold.csv gotowy do wysłania na Kaggle!
